# CEG-WM Colab 完整工作流执行指南

本 Notebook 在 Google Colab 上执行完整的机器学习工作流（embed/detect/calibrate/evaluate），生成新的 run_root 目录并下载最终结果。

**执行流水线**：
- **Embed 阶段**：对输入进行特征提取和防水印嵌入
- **Detect 阶段**：检测防水印信号
- **Calibrate 阶段**：校准检测阈值和概率
- **Evaluate 阶段**：评估整个工作流的性能

## 第 1 步：清理旧代码并拉取 GitHub 项目

从 GitHub 拉取 CEG-WM 代码到 Colab 工作目录。

**方式：自动拉取指定仓库与分支**
- 仓库：https://github.com/RICHAAARC/CEG-WM.git
- 分支：Geometry_Chain

In [ ]:
import sys
import os
from pathlib import Path
import psutil

# 检测 Google Colab 环境
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("初始化 Notebook 环境")
print("=" * 80)

# 设置工作目录
if IN_COLAB:
    WORK_DIR = Path("/tmp/ceg_wm_workspace")
    print(f"✓ 检测到 Google Colab 环境")
else:
    WORK_DIR = Path.cwd()
    print(f"✓ 本地 Jupyter 环境")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"  工作目录：{WORK_DIR}")

# 显示系统信息
print("\n系统信息：")
print(f"  操作系统：{sys.platform}")
print(f"  Python 版本：{sys.version.split()[0]}")
print(f"  CPU 核心数：{psutil.cpu_count()}")
print(f"  总内存：{psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"  可用内存：{psutil.virtual_memory().available / (1024**3):.1f} GB")

print("\n✓ 环境初始化完成")

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "Geometry_Chain"
TARGET_DIR = WORK_DIR / "CEG-WM"

print("=" * 80)
print("从 GitHub 拉取 CEG-WM 代码")
print("=" * 80)
print(f"目标仓库：{REPO_URL}")
print(f"目标分支：{REPO_BRANCH}")
print(f"工作目录：{WORK_DIR}")

if not WORK_DIR.exists():
    WORK_DIR.mkdir(parents=True, exist_ok=True)

# 1) 删除旧项目代码
print("\n[1/2] 检查并清理旧项目代码...")
candidate_dirs = []

if TARGET_DIR.exists() and TARGET_DIR.is_dir():
    candidate_dirs.append(TARGET_DIR)

for subdir in WORK_DIR.iterdir():
    if not subdir.is_dir() or subdir == TARGET_DIR:
        continue
    if (subdir / "main" / "cli").exists() and (subdir / "configs").exists() and (subdir / "scripts").exists():
        candidate_dirs.append(subdir)

# 去重并排序，保证输出稳定
unique_dirs = sorted({str(p): p for p in candidate_dirs}.values(), key=lambda p: str(p))

if unique_dirs:
    for old_dir in unique_dirs:
        print(f"  删除旧目录：{old_dir}")
        shutil.rmtree(old_dir)
    print(f"  ✓ 已删除旧目录数量：{len(unique_dirs)}")
else:
    print("  ✓ 未发现旧项目代码")

# 2) 从 GitHub 拉取指定分支
print("\n[2/2] 拉取新代码...")
if shutil.which("git") is None:
    raise RuntimeError("当前环境未检测到 git 命令，请先安装 git 后重试")

clone_cmd = [
    "git", "clone",
    "--depth", "1",
    "--branch", REPO_BRANCH,
    REPO_URL,
    str(TARGET_DIR),
]
print("执行命令：")
print("  " + " ".join(clone_cmd))

clone_result = subprocess.run(
    clone_cmd,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

if clone_result.returncode != 0:
    print("\nSTDOUT（最后 20 行）：")
    print("\n".join((clone_result.stdout or "").splitlines()[-20:]))
    print("\nSTDERR（最后 20 行）：")
    print("\n".join((clone_result.stderr or "").splitlines()[-20:]))
    raise RuntimeError(f"git clone 失败，返回码={clone_result.returncode}")

required_paths = [
    TARGET_DIR / "main" / "cli",
    TARGET_DIR / "configs",
    TARGET_DIR / "scripts",
    TARGET_DIR / "requirements.txt",
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"拉取完成但目录结构不完整，缺失：{missing_paths}")

print(f"✓ 代码拉取完成：{TARGET_DIR}")
print(f"  分支：{REPO_BRANCH}")
print("  关键目录结构校验通过")

In [ ]:

# 验证和定位 CEG-WM 代码库
from pathlib import Path

def find_ceg_wm_root(base_dir):
    """
    功能：查找 CEG-WM 根目录（智能路径发现）。
    """
    base_dir = Path(base_dir)
    
    # 检查：特征目录结构
    characteristic_dirs = ["main/cli", "configs", "scripts", "requirements.txt"]
    
    # 首先检查直接目录
    if all((base_dir / d).exists() for d in characteristic_dirs):
        return base_dir
    
    # 查找包含正确结构的子目录
    for subdir in sorted(base_dir.iterdir()):
        if subdir.is_dir() and not subdir.name.startswith('.'):
            if all((subdir / d).exists() for d in characteristic_dirs):
                return subdir
    
    # 尝试找任何包含 main/cli 的目录（最后的备选方案）
    for possible_root in base_dir.rglob("main"):
        if possible_root.is_dir():
            ceg_root = possible_root.parent
            if (ceg_root / "configs").exists() and (ceg_root / "scripts").exists():
                return ceg_root
    
    raise FileNotFoundError(
        f"无法找到 CEG-WM 根目录\n"
        f"  搜索路径：{base_dir}\n"
        f"  期望目录结构：main/cli/, configs/, scripts/, requirements.txt\n"
        f"  请检查 GitHub 拉取步骤是否执行成功"
    )

# 定位 CEG-WM 根目录（GitHub 拉取后）
try:
    CEG_WM_ROOT = find_ceg_wm_root(WORK_DIR)
    print(f"✓ 找到 CEG-WM 根目录：{CEG_WM_ROOT}")
    print(f"  绝对路径：{CEG_WM_ROOT.resolve()}")
except FileNotFoundError as e:
    print(f"✗ {e}")
    # 列出实际的目录结构以帮助调试
    print("\n实际目录结构：")
    for item in WORK_DIR.iterdir():
        if item.is_dir() and not item.name.startswith('.'):
            print(f"  {item.name}/")
            for subitem in item.iterdir():
                if subitem.is_dir():
                    print(f"    {subitem.name}/")


In [ ]:

# 添加 CEG-WM 到 Python 路径
if str(CEG_WM_ROOT) not in sys.path:
    sys.path.insert(0, str(CEG_WM_ROOT))

# 验证关键模块可导入
print("验证关键模块导入...")
try:
    from main.cli import run_embed
    print("  ✓ main.cli.run_embed 导入成功")
except ImportError as e:
    print(f"  ✗ 导入失败：{e}")
    print("    这表示 CEG-WM 依赖未完整安装，将在下一步修复")


## 第 2 步：安装依赖包

安装 CEG-WM 项目本身和所有依赖。这是关键步骤！

In [ ]:

import subprocess
import sys

print("=" * 80)
print("安装依赖包")
print("=" * 80)

# 步骤 1：升级 pip
print("\n[1/3] 升级 pip...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], 
               capture_output=True)
print("  ✓ pip 已升级")

# 步骤 2：安装系统依赖
if IN_COLAB:
    print("\n[2/3] 安装系统依赖...")
    subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"], 
                   capture_output=True)
    print("  ✓ 系统依赖已安装")

# 步骤 3：安装 CEG-WM 项目本身（关键！）
print("\n[3/3] 安装 CEG-WM 项目...")
try:
    # 方式 A：从 pyproject.toml（推荐）
    pyproject_file = CEG_WM_ROOT / "pyproject.toml"
    requirements_file = CEG_WM_ROOT / "requirements.txt"
    
    if pyproject_file.exists():
        print(f"  从 pyproject.toml 安装：{pyproject_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", str(CEG_WM_ROOT)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 项目安装成功（开发模式）")
        else:
            print(f"  ⚠ 项目安装失败，尝试备选方案")
            print(f"    错误：{result.stderr[-200:]}")
    
    elif requirements_file.exists():
        print(f"  从 requirements.txt 安装：{requirements_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 依赖安装成功")
        else:
            print(f"  ⚠ 部分依赖安装失败，继续使用...")
    else:
        print("  ⚠ 未找到 pyproject.toml 或 requirements.txt")
        
except subprocess.TimeoutExpired:
    print("  ⚠ 安装超时（300 秒），但继续执行")
except Exception as e:
    print(f"  ⚠ 安装出错：{e}，但继续执行")

print("\n✓ 依赖安装步骤完成")


## 第 3 步：下载模型权重

真实下载 Stable Diffusion 3 模型。这是关键步骤！

In [ ]:
import time
import os
import gc
import torch
from pathlib import Path
from huggingface_hub import scan_cache_dir, snapshot_download

print("=" * 80)
print("下载模型权重")
print("=" * 80)

# 配置模型缓存目录
MODEL_CACHE_DIR = WORK_DIR / "huggingface_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE_DIR)

print(f"\n模型缓存目录：{MODEL_CACHE_DIR}")

print("\n检查 Hugging Face 认证...")
try:
    from huggingface_hub import HfApi
    api = HfApi()
    try:
        user_info = api.whoami()
        print(f"  ✓ 已认证：{user_info.get('name', 'Unknown')}")
    except Exception:
        print("  ℹ 未认证（使用匿名访问，私有/受限模型可能无法下载）")
except ImportError:
    print("  ⚠ huggingface_hub 未安装")

print("\n" + "=" * 80)
print("下载 Stable Diffusion 3.5 模型")
print("=" * 80)

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
print(f"目标模型：{MODEL_ID}")

def check_model_cached(model_id):
    """
    功能：检查模型是否已在缓存中。
    
    Check if model is already cached locally.
    
    Args:
        model_id: Model identifier.
        
    Returns:
        Boolean indicating if model is cached.
    """
    try:
        cache_info = scan_cache_dir()
        for repo in cache_info.repos:
            if model_id in repo.repo_id:
                return True
        return False
    except Exception:
        return False

if check_model_cached(MODEL_ID):
    print(f"✓ 模型已缓存：{MODEL_ID}")
    print("  跳过下载")
else:
    print(f"\n⏳ 模型未缓存，开始下载：{MODEL_ID}")
    print("   这可能需要 5-20 分钟（取决于网络与镜像）")
    
    try:
        snapshot_path = snapshot_download(
            repo_id=MODEL_ID,
            cache_dir=str(MODEL_CACHE_DIR),
            local_files_only=False,
        )
        print("  ✓ 模型下载完成")
        print(f"  本地快照路径：{snapshot_path}")
    except Exception as e:
        error_msg = str(e)
        print(f"  ✗ 模型下载失败：{e}")
        
        if "404" in error_msg or "Entry Not Found" in error_msg:
            print("\n  ❌ 错误：无法访问模型（404）")
            print("  原因：SD3.5 可能需要先在 Hugging Face 页面授权")
        elif "401" in error_msg or "Unauthorized" in error_msg or "403" in error_msg:
            print("\n  ❌ 错误：认证失败（401/403）")
            print("  原因：HF_TOKEN 无效、缺失或未获模型访问权限")
        else:
            print("\n  解决方案：")
            print("    1. 检查网络连接")
            print("    2. 检查 HF_TOKEN 与模型访问授权")
            print("    3. 重新执行本 cell")
        
        print("\n  ⚠️ 继续执行（后续步骤可能失败）...")

print("\n清理内存...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("  ✓ GPU 缓存已清理")

print("✓ 模型准备完成")

## 第 4 步：配置选择和数据准备

选择工作流配置并准备输入数据。

In [ ]:
import json

print("=" * 80)
print("工作流配置和数据准备")
print("=" * 80)

# 选择配置文件（对齐当前仓库，仅保留 default）
CONFIG_CHOICE = "default"
print(f"\n选定配置：{CONFIG_CHOICE}")

CONFIG_FILE = CEG_WM_ROOT / "configs" / f"{CONFIG_CHOICE}.yaml"
if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"默认配置不存在：{CONFIG_FILE}")

PAPER_SPEC_FILE = CEG_WM_ROOT / "configs" / "paper_faithfulness_spec.yaml"
if PAPER_SPEC_FILE.exists():
    print(f"  ✓ 论文一致性规范存在：{PAPER_SPEC_FILE.name}")
else:
    print(f"  ⚠ 未找到论文一致性规范：{PAPER_SPEC_FILE}")

MODEL_CFG_FILE = CEG_WM_ROOT / "configs" / "model_sd3.yaml"
if MODEL_CFG_FILE.exists():
    print(f"  ✓ SD3 模型配置存在：{MODEL_CFG_FILE.name}")
else:
    print(f"  ⚠ 未找到 SD3 模型配置：{MODEL_CFG_FILE}")

print(f"  配置文件路径：{CONFIG_FILE}")
print(f"  配置文件大小：{CONFIG_FILE.stat().st_size / 1024:.1f} KB")

# 创建数据目录
data_dir = CEG_WM_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# 准备输入数据
print("\n准备输入数据...")

In [ ]:
# 第 4.5 步：paper_full_cuda 运行前预检（建议在第 5 步前执行）
import os
import shutil
import socket
from pathlib import Path

print("=" * 80)
print("运行前预检：paper_full_cuda")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals():
    raise RuntimeError("请先执行前面的单元，确保 CEG_WM_ROOT 已初始化")

precheck_results = []

def _record_check(name: str, ok: bool, detail: str):
    precheck_results.append({"name": name, "ok": ok, "detail": detail})
    status = "✓" if ok else "✗"
    print(f"{status} {name}: {detail}")

# 1) CUDA / GPU 预检
try:
    import torch
    cuda_ok = bool(torch.cuda.is_available())
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        _record_check("CUDA 可用", True, f"device={gpu_name}")
    else:
        _record_check("CUDA 可用", False, "未检测到 GPU，请在 Colab 切换到 GPU Runtime")
except Exception as exc:
    _record_check("CUDA 可用", False, f"torch 检查异常: {type(exc).__name__}: {exc}")

# 2) 关键配置与脚本存在性预检
required_paths = [
    CEG_WM_ROOT / "configs" / "paper_full_cuda.yaml",
    CEG_WM_ROOT / "configs" / "paper_faithfulness_spec.yaml",
    CEG_WM_ROOT / "scripts" / "run_onefile_workflow.py",
    CEG_WM_ROOT / "scripts" / "assert_paper_mechanisms.py",
]
for path in required_paths:
    _record_check(f"文件存在: {path.name}", path.exists(), str(path))

# 3) Python 包可用性预检
required_modules = ["yaml", "huggingface_hub", "diffusers", "transformers", "safetensors"]
for module_name in required_modules:
    try:
        __import__(module_name)
        _record_check(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        _record_check(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

# 4) Hugging Face 网络与访问预检（非阻塞，但强烈建议通过）
hf_ok = False
hf_detail = "未执行"
try:
    from huggingface_hub import HfApi
    socket.setdefaulttimeout(15)
    api = HfApi()
    _ = api.model_info("stabilityai/stable-diffusion-3.5-medium")
    hf_ok = True
    hf_detail = "可访问 stabilityai/stable-diffusion-3.5-medium"
except Exception as exc:
    hf_ok = False
    hf_detail = f"访问失败（可能是网络/权限问题）: {type(exc).__name__}: {exc}"
_record_check("HF 模型可访问", hf_ok, hf_detail)

# 5) 工作空间磁盘空间预检
usage = shutil.disk_usage(str(CEG_WM_ROOT))
free_gb = usage.free / (1024 ** 3)
disk_ok = free_gb >= 20.0
_record_check("磁盘剩余空间", disk_ok, f"free={free_gb:.1f} GB，建议 >= 20 GB")

# 汇总
hard_fail_names = {
    "CUDA 可用",
    "文件存在: paper_full_cuda.yaml",
    "文件存在: run_onefile_workflow.py",
    "模块可导入: diffusers",
    "模块可导入: transformers",
}
hard_fail = [item for item in precheck_results if (item["name"] in hard_fail_names and not item["ok"]) ]
soft_fail = [item for item in precheck_results if (item["name"] not in hard_fail_names and not item["ok"]) ]

print("\n" + "-" * 80)
print("预检结果汇总")
print("-" * 80)
print(f"硬性失败项数量: {len(hard_fail)}")
print(f"软性失败项数量: {len(soft_fail)}")

if hard_fail:
    print("\n✗ 预检未通过（存在硬性失败项），请先修复后再执行第 5 步。")
    for item in hard_fail:
        print(f"  - {item['name']}: {item['detail']}")
else:
    if soft_fail:
        print("\n✓ 预检通过（存在软性风险项，建议优先处理）：")
        for item in soft_fail:
            print(f"  - {item['name']}: {item['detail']}")
    else:
        print("\n✓ 预检全部通过，可以执行第 5 步。")

## 第 5-8 步：CUDA 真实模型一键完整流程（含审计与签署）

### 第 5 步
- 一键执行完整链路（embed → detect → calibrate → evaluate → audits → signoff）

### 第 6 步
- 如需单独复跑审计脚本，直接调用 `scripts/run_all_audits.py --strict`

### 第 7 步
- 如需单独复跑签署脚本，直接调用 `scripts/run_freeze_signoff.py`

### 第 8 步
- 汇总显示关键输出文件与阶段状态

**执行策略**：

- 强制要求 CUDA 可用
- 强制使用真实模型 `stabilityai/stable-diffusion-3.5-medium`
- 主流程优先调用 `scripts/run_onefile_workflow.py`，降低 Notebook 内多段逻辑出错概率

In [ ]:
import json
import sys
import subprocess
import shutil
from pathlib import Path
import torch

print("=" * 80)
print("第 5 步：CUDA + paper_full_cuda 一键完整 workflow（含 audits/signoff）")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'CONFIG_CHOICE' not in globals():
    raise RuntimeError("请先执行前面的环境与配置 cell（第 1-4 步）")

if not torch.cuda.is_available():
    raise RuntimeError("当前运行时未检测到 CUDA。请在 Colab 切换到 GPU Runtime 后重试。")

gpu_name = torch.cuda.get_device_name(0)
print(f"✓ CUDA 可用，设备：{gpu_name}")

RUN_ROOT = CEG_WM_ROOT / "outputs" / "colab_run_paper_full_cuda"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("\n准备新的运行目录...")
for name in ["records", "artifacts", "logs"]:
    target = RUN_ROOT / name
    if target.exists():
        shutil.rmtree(target)
        print(f"  已清理：{target}")

logs_dir = RUN_ROOT / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)

PROJECT_PAPER_CFG = CEG_WM_ROOT / "configs" / "paper_full_cuda.yaml"
if not PROJECT_PAPER_CFG.exists():
    raise FileNotFoundError(
        f"未找到项目配置：{PROJECT_PAPER_CFG}。"
        "请确认仓库已包含 configs/paper_full_cuda.yaml。"
    )

ACTIVE_CONFIG_FILE = PROJECT_PAPER_CFG
ACTIVE_WORKFLOW_PROFILE = "paper_full_cuda"
ACTIVE_SIGNOFF_PROFILE = "paper"

print("\n一键执行参数：")
print(f"  输出目录：{RUN_ROOT}")
print(f"  运行配置（项目内）：{ACTIVE_CONFIG_FILE}")
print(f"  profile: {ACTIVE_WORKFLOW_PROFILE}")
print(f"  signoff-profile: {ACTIVE_SIGNOFF_PROFILE}")

command = [
    sys.executable,
    "scripts/run_onefile_workflow.py",
    "--cfg",
    str(ACTIVE_CONFIG_FILE),
    "--run-root",
    str(RUN_ROOT),
    "--profile",
    ACTIVE_WORKFLOW_PROFILE,
    "--signoff-profile",
    ACTIVE_SIGNOFF_PROFILE,
]

print("\n执行命令：")
print("  " + " ".join(command))

result = subprocess.run(
    command,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

workflow_log = logs_dir / "workflow_execution.log"
with open(workflow_log, "w", encoding="utf-8") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(command))
    f.write("\n\nRETURN_CODE:\n")
    f.write(str(result.returncode))
    f.write("\n\nSTDOUT:\n")
    f.write(result.stdout or "")
    f.write("\n\nSTDERR:\n")
    f.write(result.stderr or "")

print(f"\n返回码：{result.returncode}")
print(f"日志文件：{workflow_log}")

if result.stdout:
    print("\nSTDOUT（最后 40 行）：")
    print("\n".join(result.stdout.splitlines()[-40:]))
if result.stderr:
    print("\nSTDERR（最后 20 行）：")
    print("\n".join(result.stderr.splitlines()[-20:]))

workflow_recovered = False

if result.returncode != 0:
    failed_step = "<unknown>"
    for line in (result.stdout or "").splitlines():
        if line.startswith("[onefile] step=") and " end=" in line and "return_code=" in line:
            step_name = line.split("step=", 1)[1].split(" ", 1)[0]
            step_return_code = line.rsplit("return_code=", 1)[1].strip()
            if step_return_code != "0":
                failed_step = step_name

    print("\n失败诊断（自动）...")
    print(f"  识别到失败阶段：{failed_step}")

    if failed_step == "multi_protocol_evaluation":
        compare_summary_path = RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json"
        print(f"  compare_summary 路径：{compare_summary_path}")
        print(f"  compare_summary 存在：{compare_summary_path.exists()}")
        print("  提示：该失败通常来自某个 protocol 子运行失败触发 fail-fast。")
        print("  请确认 onefile 脚本已包含 --continue-on-fail，或更新到最新代码后重跑第 5 步。")

    elif failed_step == "assert_paper_mechanisms":
        assert_cfg_path = RUN_ROOT / "artifacts" / "workflow_cfg" / f"profile_{ACTIVE_WORKFLOW_PROFILE}.yaml"
        if not assert_cfg_path.exists():
            assert_cfg_path = ACTIVE_CONFIG_FILE

        assert_cmd = [
            sys.executable,
            "scripts/assert_paper_mechanisms.py",
            "--run-root",
            str(RUN_ROOT),
            "--config",
            str(assert_cfg_path),
            "--profile",
            ACTIVE_WORKFLOW_PROFILE,
        ]
        print("  进入 assert 专项诊断...")
        print("  复跑命令：")
        print("    " + " ".join(assert_cmd))

        assert_result = subprocess.run(
            assert_cmd,
            cwd=str(CEG_WM_ROOT),
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )

        assert_debug_log = RUN_ROOT / "logs" / "assert_paper_mechanisms_debug.log"
        with open(assert_debug_log, "w", encoding="utf-8") as f:
            f.write("COMMAND:\n")
            f.write(" ".join(assert_cmd))
            f.write("\n\nRETURN_CODE:\n")
            f.write(str(assert_result.returncode))
            f.write("\n\nSTDOUT:\n")
            f.write(assert_result.stdout or "")
            f.write("\n\nSTDERR:\n")
            f.write(assert_result.stderr or "")

        print(f"  assert 诊断日志：{assert_debug_log}")
        if assert_result.stdout:
            print("\n  assert STDOUT（最后 80 行）：")
            print("\n".join(assert_result.stdout.splitlines()[-80:]))
        if assert_result.stderr:
            print("\n  assert STDERR（最后 40 行）：")
            print("\n".join(assert_result.stderr.splitlines()[-40:]))

        failure_lines = [
            line.strip() for line in (assert_result.stdout or "").splitlines()
            if line.strip().startswith("-")
        ]
        if failure_lines:
            print("\n  assert 失败条目：")
            for item in failure_lines:
                print(f"    {item}")

        if (
            "detect content evidence must include sync_digest" in (assert_result.stdout or "")
            or "detect content evidence must include anchor_digest" in (assert_result.stdout or "")
            or "geometry anchor_metrics must exist" in (assert_result.stdout or "")
        ):
            print("\n  提示：若你的代码包未包含最新断言兼容修复，可能会误报几何锚点缺失。")
            print("  建议：更新 scripts/assert_paper_mechanisms.py 后重跑第 5 步。")

    elif failed_step == "signoff":
        signoff_report_path = RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json"
        print(f"  signoff_report 路径：{signoff_report_path}")
        print(f"  signoff_report 存在：{signoff_report_path.exists()}")

        should_autofix_matrix = False
        if signoff_report_path.exists():
            try:
                with open(signoff_report_path, "r", encoding="utf-8") as f:
                    signoff_obj = json.load(f)

                freeze_decision = signoff_obj.get("freeze_signoff_decision", "<absent>")
                signoff_profile = signoff_obj.get("signoff_profile", "<absent>")
                require_experiment_matrix = signoff_obj.get("require_experiment_matrix", "<absent>")
                print(f"  freeze_signoff_decision: {freeze_decision}")
                print(f"  signoff_profile: {signoff_profile}")
                print(f"  require_experiment_matrix: {require_experiment_matrix}")

                summary_obj = signoff_obj.get("summary") if isinstance(signoff_obj, dict) else None
                if isinstance(summary_obj, dict):
                    print(f"  summary.static_audit_counts: {summary_obj.get('static_audit_counts', '<absent>')}")
                    print(f"  summary.evidence_status: {summary_obj.get('evidence_status', '<absent>')}")

                blocking_reasons = signoff_obj.get("blocking_reasons") if isinstance(signoff_obj, dict) else None
                if isinstance(blocking_reasons, list) and blocking_reasons:
                    print("\n  signoff 阻断原因（最多显示前 10 条）：")
                    for idx, reason in enumerate(blocking_reasons[:10], start=1):
                        if isinstance(reason, dict):
                            source = reason.get("source", "<absent>")
                            audit_id = reason.get("audit_id", "<absent>")
                            rule = reason.get("rule", "<absent>")
                            print(f"    [{idx}] source={source} audit_id={audit_id}")
                            print(f"         rule={rule}")
                            if isinstance(reason.get("errors"), list) and reason.get("errors"):
                                print(f"         errors={reason.get('errors')[:3]}")
                        else:
                            print(f"    [{idx}] {reason}")

                audits_obj = signoff_obj.get("audits") if isinstance(signoff_obj, dict) else None
                block_fail_audit_ids = []
                if isinstance(audits_obj, list):
                    block_fails = [
                        item for item in audits_obj
                        if isinstance(item, dict)
                        and item.get("result") == "FAIL"
                        and item.get("severity") == "BLOCK"
                    ]
                    if block_fails:
                        print(f"\n  BLOCK 级失败审计数量：{len(block_fails)}（最多显示前 8 条）")
                        for item in block_fails[:8]:
                            audit_id = item.get('audit_id', '<absent>')
                            block_fail_audit_ids.append(audit_id)
                            print(f"    - {audit_id}: {item.get('rule', '<absent>')}")
                            evidence_obj = item.get("evidence") if isinstance(item, dict) else None
                            if isinstance(evidence_obj, dict):
                                stderr_tail = (evidence_obj.get("stderr_text") or "").strip().splitlines()
                                stdout_tail = (evidence_obj.get("stdout_text") or "").strip().splitlines()
                                if stderr_tail:
                                    print("      stderr（最后 6 行）：")
                                    for line in stderr_tail[-6:]:
                                        print(f"        {line}")
                                if stdout_tail:
                                    print("      stdout（最后 6 行）：")
                                    for line in stdout_tail[-6:]:
                                        print(f"        {line}")

                        normalized_ids = {str(x) for x in block_fail_audit_ids}
                        matrix_related_ids = {
                            "ERROR.audit_experiment_matrix_outputs_schema",
                            "experiment_matrix.outputs_schema",
                        }
                        if normalized_ids and normalized_ids.issubset(matrix_related_ids):
                            should_autofix_matrix = True

                run_root_evidence = signoff_obj.get("run_root_evidence") if isinstance(signoff_obj, dict) else None
                if isinstance(run_root_evidence, dict):
                    evidence_status = run_root_evidence.get("status", "<absent>")
                    evidence_errors = run_root_evidence.get("errors", [])
                    print(f"\n  run_root_evidence.status: {evidence_status}")
                    if isinstance(evidence_errors, list) and evidence_errors:
                        print(f"  run_root_evidence.errors（最多前 5 条）：{evidence_errors[:5]}")

                if freeze_decision != "ALLOW_FREEZE":
                    print("\n  提示：signoff 非 ALLOW_FREEZE 时，优先按 blocking_reasons 与 run_root_evidence.errors 修复。")

            except Exception as exc:
                print(f"  读取 signoff_report 失败：{type(exc).__name__}: {exc}")
        else:
            print("  signoff_report.json 不存在，请优先检查 signoff 步骤 stderr 与 workflow_execution.log。")

        if should_autofix_matrix:
            print("\n  命中自动修复条件：仅 experiment matrix 审计阻断，开始补跑 matrix 并重试 signoff...")

            matrix_batch_root = RUN_ROOT / "outputs" / "experiment_matrix"
            matrix_cfg_autofix_path = RUN_ROOT / "artifacts" / "workflow_cfg" / "matrix_autofix_runtime.yaml"
            generated_profile_cfg = RUN_ROOT / "artifacts" / "workflow_cfg" / f"profile_{ACTIVE_WORKFLOW_PROFILE}.yaml"
            matrix_base_cfg = generated_profile_cfg if generated_profile_cfg.exists() else ACTIVE_CONFIG_FILE
            matrix_config_for_run = matrix_base_cfg
            print(f"  matrix 基础配置：{matrix_base_cfg}")

            try:
                import yaml
                with open(matrix_base_cfg, "r", encoding="utf-8") as f:
                    matrix_cfg_obj = yaml.safe_load(f) or {}

                def _strip_ablation_normalized(node):
                    removed = 0
                    if isinstance(node, dict):
                        for key, value in list(node.items()):
                            if key == "ablation" and isinstance(value, dict) and "normalized" in value:
                                value.pop("normalized", None)
                                removed += 1
                            removed += _strip_ablation_normalized(value)
                    elif isinstance(node, list):
                        for item in node:
                            removed += _strip_ablation_normalized(item)
                    return removed

                removed_count = _strip_ablation_normalized(matrix_cfg_obj)
                if isinstance(matrix_cfg_obj, dict):
                    matrix_cfg_autofix_path.parent.mkdir(parents=True, exist_ok=True)
                    with open(matrix_cfg_autofix_path, "w", encoding="utf-8") as f:
                        yaml.safe_dump(matrix_cfg_obj, f, allow_unicode=True, sort_keys=False)
                    matrix_config_for_run = matrix_cfg_autofix_path
                    print("  已生成 matrix 自动修复配置（递归移除 ablation.normalized）")
                    print(f"    removed_ablation_normalized_count={removed_count}")
                    print(f"    runtime_cfg={matrix_config_for_run}")
            except Exception as exc:
                print(f"  生成 matrix 自动修复配置失败，回退使用原配置：{type(exc).__name__}: {exc}")

            matrix_cmd = [
                sys.executable,
                "scripts/run_experiment_matrix.py",
                "--config",
                str(matrix_config_for_run),
                "--batch-root",
                str(matrix_batch_root),
            ]
            print("  matrix 命令：")
            print("    " + " ".join(matrix_cmd))

            matrix_result = subprocess.run(
                matrix_cmd,
                cwd=str(CEG_WM_ROOT),
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
            )

            matrix_log = RUN_ROOT / "logs" / "experiment_matrix_autofix.log"
            with open(matrix_log, "w", encoding="utf-8") as f:
                f.write("COMMAND:\n")
                f.write(" ".join(matrix_cmd))
                f.write("\n\nRETURN_CODE:\n")
                f.write(str(matrix_result.returncode))
                f.write("\n\nSTDOUT:\n")
                f.write(matrix_result.stdout or "")
                f.write("\n\nSTDERR:\n")
                f.write(matrix_result.stderr or "")
            print(f"  matrix 日志：{matrix_log}")
            print(f"  matrix 返回码：{matrix_result.returncode}")

            if matrix_result.stderr:
                print("  matrix STDERR（最后 20 行）：")
                print("\n".join(matrix_result.stderr.splitlines()[-20:]))

            if matrix_result.returncode == 0:
                signoff_retry_cmd = [
                    sys.executable,
                    "scripts/run_freeze_signoff.py",
                    "--run-root",
                    str(RUN_ROOT),
                    "--repo-root",
                    str(CEG_WM_ROOT),
                    "--signoff-profile",
                    ACTIVE_SIGNOFF_PROFILE,
                    "--require-experiment-matrix",
                ]
                print("\n  重试 signoff 命令：")
                print("    " + " ".join(signoff_retry_cmd))

                signoff_retry_result = subprocess.run(
                    signoff_retry_cmd,
                    cwd=str(CEG_WM_ROOT),
                    capture_output=True,
                    text=True,
                    encoding="utf-8",
                    errors="replace",
                )

                signoff_retry_log = RUN_ROOT / "logs" / "signoff_retry_after_matrix.log"
                with open(signoff_retry_log, "w", encoding="utf-8") as f:
                    f.write("COMMAND:\n")
                    f.write(" ".join(signoff_retry_cmd))
                    f.write("\n\nRETURN_CODE:\n")
                    f.write(str(signoff_retry_result.returncode))
                    f.write("\n\nSTDOUT:\n")
                    f.write(signoff_retry_result.stdout or "")
                    f.write("\n\nSTDERR:\n")
                    f.write(signoff_retry_result.stderr or "")
                print(f"  signoff 重试日志：{signoff_retry_log}")
                print(f"  signoff 重试返回码：{signoff_retry_result.returncode}")

                if signoff_retry_result.returncode == 0:
                    workflow_recovered = True
                    print("\n  ✓ 自动修复成功：matrix 已补跑，signoff 重试通过。")
                else:
                    print("\n  ✗ 自动修复未成功：signoff 重试仍失败，请查看 signoff_retry_after_matrix.log。")
                    if signoff_retry_result.stderr:
                        print("  signoff 重试 STDERR（最后 20 行）：")
                        print("\n".join(signoff_retry_result.stderr.splitlines()[-20:]))
            else:
                print("\n  ✗ 自动修复未成功：experiment matrix 补跑失败，请查看 experiment_matrix_autofix.log。")

    elif failed_step in {"detect", "calibrate", "evaluate"}:
        detect_record_path = RUN_ROOT / "records" / "detect_record.json"
        print(f"  detect_record 路径：{detect_record_path}")
        if detect_record_path.exists():
            try:
                with open(detect_record_path, "r", encoding="utf-8") as f:
                    detect_record_obj = json.load(f)
                content_payload = detect_record_obj.get("content_evidence_payload") if isinstance(detect_record_obj, dict) else None
                if isinstance(content_payload, dict):
                    content_status = content_payload.get("status", "<absent>")
                    content_score = content_payload.get("score", None)
                    content_failure_reason = content_payload.get("content_failure_reason", "<absent>")
                    content_mismatch_reason = content_payload.get("content_mismatch_reason", "<absent>")
                    print(f"  content_evidence_payload.status: {content_status}")
                    print(f"  content_evidence_payload.score: {content_score}")
                    print(f"  content_failure_reason: {content_failure_reason}")
                    print(f"  content_mismatch_reason: {content_mismatch_reason}")
                    if content_status != "ok":
                        print("  提示：calibrate 仅接受 status=ok 且 score 为数值的样本。")
                        print("  请先修复 detect 阶段产生 mismatch/absent 的根因，再重跑第 5 步。")
                else:
                    print("  detect_record 缺少 content_evidence_payload 字段")
            except Exception as exc:
                print(f"  诊断读取失败：{type(exc).__name__}: {exc}")
        else:
            print("  detect_record.json 不存在，说明流程在 detect 前已失败。")

    else:
        print("  未命中专项诊断分支，请优先查看 workflow_execution.log 全量输出。")

    if not workflow_recovered:
        raise RuntimeError(f"onefile workflow 执行失败，return_code={result.returncode}。请查看日志：{workflow_log}")
    print("\n已通过自动修复恢复 workflow，继续后续步骤。")

STAGE_STATUS = {}
STAGE_STATUS["embed"] = "ok" if (RUN_ROOT / "records" / "embed_record.json").exists() else "failed"
STAGE_STATUS["detect"] = "ok" if (RUN_ROOT / "records" / "detect_record.json").exists() else "failed"
STAGE_STATUS["calibrate"] = "ok" if (RUN_ROOT / "records" / "calibration_record.json").exists() else "failed"
STAGE_STATUS["evaluate"] = "ok" if (RUN_ROOT / "records" / "evaluate_record.json").exists() else "failed"
STAGE_STATUS["multi_protocol"] = "ok" if (RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json").exists() else "failed"
STAGE_STATUS["signoff"] = "ok" if (RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json").exists() else "failed"

REQUIRED_PACKAGE_FILES = [
    RUN_ROOT / "records" / "embed_record.json",
    RUN_ROOT / "records" / "detect_record.json",
    RUN_ROOT / "records" / "calibration_record.json",
    RUN_ROOT / "records" / "evaluate_record.json",
    RUN_ROOT / "artifacts" / "evaluation_report.json",
    RUN_ROOT / "artifacts" / "run_closure.json",
    RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json",
    RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json",
]

SIGNOFF_DECISION = "<absent>"
signoff_path = RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json"
if signoff_path.exists():
    try:
        with open(signoff_path, "r", encoding="utf-8") as f:
            _signoff_obj = json.load(f)
        SIGNOFF_DECISION = _signoff_obj.get("freeze_signoff_decision", "<absent>")
    except Exception:
        SIGNOFF_DECISION = "<parse_failed>"

EMBED_STATUS = STAGE_STATUS.get("embed", "unknown")
DETECT_STATUS = STAGE_STATUS.get("detect", "unknown")
CALIBRATE_STATUS = STAGE_STATUS.get("calibrate", "unknown")
EVALUATE_STATUS = STAGE_STATUS.get("evaluate", "unknown")

EMBED_RUNTIME_FALLBACK_USED = False

print("\n" + "=" * 80)
print("一键 workflow 执行完成")
print("=" * 80)
print(f"阶段汇总：{STAGE_STATUS}")
print(f"Signoff 决策：{SIGNOFF_DECISION}")
print(f"结果目录：{RUN_ROOT}")

In [ ]:
import json
import subprocess
import sys
import shutil
from pathlib import Path

print("=" * 80)
print("第 5.5 步：evaluation_report 语义校验 + coverage 审计可执行性检查")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，确保 CEG_WM_ROOT 和 RUN_ROOT 已初始化")

evaluation_report_path = RUN_ROOT / "artifacts" / "evaluation_report.json"
if not evaluation_report_path.exists():
    raise FileNotFoundError(f"未找到 evaluation_report.json: {evaluation_report_path}")

with open(evaluation_report_path, "r", encoding="utf-8") as f:
    raw_eval_report_obj = json.load(f)

if not isinstance(raw_eval_report_obj, dict):
    raise RuntimeError("evaluation_report.json 必须是 JSON object")

if isinstance(raw_eval_report_obj.get("evaluation_report"), dict):
    eval_report_obj = raw_eval_report_obj["evaluation_report"]
else:
    eval_report_obj = raw_eval_report_obj

required_anchor_fields = [
    "cfg_digest",
    "plan_digest",
    "thresholds_digest",
    "threshold_metadata_digest",
    "impl_digest",
    "fusion_rule_version",
    "attack_protocol_version",
    "attack_protocol_digest",
    "policy_path",
]

def _get_anchor_value(report_obj, key):
    if isinstance(report_obj.get(key), str) and report_obj.get(key):
        return report_obj.get(key)
    anchors = report_obj.get("anchors")
    if isinstance(anchors, dict):
        val = anchors.get(key)
        if isinstance(val, str) and val:
            return val
    return None

missing_or_absent = []
for field in required_anchor_fields:
    value = _get_anchor_value(eval_report_obj, field)
    if value is None or value == "<absent>":
        missing_or_absent.append(field)

if eval_report_obj.get("report_type") != "evaluation_report":
    raise RuntimeError(
        f"report_type 非法，期望 evaluation_report，实际: {eval_report_obj.get('report_type')}"
    )

metrics_by_condition = eval_report_obj.get("metrics_by_attack_condition")
if not isinstance(metrics_by_condition, list):
    raise RuntimeError("metrics_by_attack_condition 必须为 list")

group_key_missing = []
for idx, item in enumerate(metrics_by_condition):
    if not isinstance(item, dict):
        group_key_missing.append(f"idx={idx}: not dict")
        continue
    group_key = item.get("group_key")
    if not isinstance(group_key, str) or not group_key:
        group_key_missing.append(f"idx={idx}: group_key missing")

if missing_or_absent:
    raise RuntimeError(f"evaluation_report 缺失或占位锚点字段: {missing_or_absent}")
if group_key_missing:
    raise RuntimeError(f"metrics_by_attack_condition 非法条目: {group_key_missing[:5]}")

print(f"✓ evaluation_report 语义校验通过: {evaluation_report_path}")
print(f"  metrics_by_attack_condition 条目数: {len(metrics_by_condition)}")

# coverage 审计脚本当前按固定路径查找报告；为 Colab run_root 增加兼容镜像
coverage_bridge_path = CEG_WM_ROOT / "outputs" / "smoke_detect" / "evaluation_report.json"
coverage_bridge_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(evaluation_report_path, coverage_bridge_path)
print(f"✓ 已写入 coverage 审计兼容镜像: {coverage_bridge_path}")

coverage_audit_cmd = [
    sys.executable,
    "scripts/audits/audit_attack_protocol_report_coverage.py",
    str(CEG_WM_ROOT),
]

coverage_result = subprocess.run(
    coverage_audit_cmd,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("\ncoverage 审计命令：")
print("  " + " ".join(coverage_audit_cmd))
print(f"coverage 审计返回码: {coverage_result.returncode}")

coverage_audit_log = RUN_ROOT / "logs" / "coverage_audit.log"
coverage_audit_log.parent.mkdir(parents=True, exist_ok=True)
with open(coverage_audit_log, "w", encoding="utf-8") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(coverage_audit_cmd))
    f.write("\n\nRETURN_CODE:\n")
    f.write(str(coverage_result.returncode))
    f.write("\n\nSTDOUT:\n")
    f.write(coverage_result.stdout or "")
    f.write("\n\nSTDERR:\n")
    f.write(coverage_result.stderr or "")

print(f"coverage 审计日志: {coverage_audit_log}")

coverage_json = None
if coverage_result.stdout and coverage_result.stdout.strip():
    try:
        coverage_json = json.loads(coverage_result.stdout)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"coverage 审计输出不是有效 JSON: {exc}")
else:
    raise RuntimeError("coverage 审计未输出 JSON")

coverage_status = coverage_json.get("result", "<absent>")
print(f"coverage 审计结果: {coverage_status}")

if coverage_status == "N.A.":
    raise RuntimeError(
        "coverage 审计返回 N.A.，说明未有效绑定 evaluation_report；请检查第 5.5 步兼容镜像与执行路径"
    )
if coverage_status != "PASS":
    missed = coverage_json.get("evidence", {}).get("missed_conditions", [])
    extra = coverage_json.get("evidence", {}).get("extra_reported_conditions", [])
    raise RuntimeError(
        f"coverage 审计未通过，missed_conditions={missed}, extra_reported_conditions={extra}"
    )

COVERAGE_AUDIT_RESULT = coverage_json
print("✓ coverage 审计通过（PASS）")

In [ ]:
print("=" * 80)
print("第 6 步：单独复跑严格审计（可选）")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再复跑审计")

audit_cmd = [
    sys.executable,
    "scripts/run_all_audits.py",
    "--repo-root",
    str(CEG_WM_ROOT),
    "--strict",
]

print("执行命令：")
print("  " + " ".join(audit_cmd))

audit_result = subprocess.run(
    audit_cmd,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

audit_log = RUN_ROOT / "logs" / "audits_strict_rerun.log"
with open(audit_log, "w", encoding="utf-8") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(audit_cmd))
    f.write("\n\nRETURN_CODE:\n")
    f.write(str(audit_result.returncode))
    f.write("\n\nSTDOUT:\n")
    f.write(audit_result.stdout or "")
    f.write("\n\nSTDERR:\n")
    f.write(audit_result.stderr or "")

print(f"返回码：{audit_result.returncode}")
print(f"日志文件：{audit_log}")

if audit_result.stdout:
    print("\nSTDOUT（最后 30 行）：")
    print("\n".join(audit_result.stdout.splitlines()[-30:]))
if audit_result.stderr:
    print("\nSTDERR（最后 20 行）：")
    print("\n".join(audit_result.stderr.splitlines()[-20:]))

In [ ]:
print("=" * 80)
print("第 7 步：单独复跑签署脚本（可选）")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再复跑签署")

signoff_cmd = [
    sys.executable,
    "scripts/run_freeze_signoff.py",
    "--run-root",
    str(RUN_ROOT),
    "--repo-root",
    str(CEG_WM_ROOT),
    "--signoff-profile",
    "paper",
]

print("执行命令：")
print("  " + " ".join(signoff_cmd))

signoff_result = subprocess.run(
    signoff_cmd,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

signoff_log = RUN_ROOT / "logs" / "signoff_rerun.log"
with open(signoff_log, "w", encoding="utf-8") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(signoff_cmd))
    f.write("\n\nRETURN_CODE:\n")
    f.write(str(signoff_result.returncode))
    f.write("\n\nSTDOUT:\n")
    f.write(signoff_result.stdout or "")
    f.write("\n\nSTDERR:\n")
    f.write(signoff_result.stderr or "")

print(f"返回码：{signoff_result.returncode}")
print(f"日志文件：{signoff_log}")

if signoff_result.stdout:
    print("\nSTDOUT（最后 30 行）：")
    print("\n".join(signoff_result.stdout.splitlines()[-30:]))
if signoff_result.stderr:
    print("\nSTDERR（最后 20 行）：")
    print("\n".join(signoff_result.stderr.splitlines()[-20:]))

In [ ]:
print("=" * 80)
print("第 8 步：关键结果速查")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步")

check_items = [
    ("embed_record", RUN_ROOT / "records" / "embed_record.json"),
    ("detect_record", RUN_ROOT / "records" / "detect_record.json"),
    ("calibration_record", RUN_ROOT / "records" / "calibration_record.json"),
    ("evaluate_record", RUN_ROOT / "records" / "evaluate_record.json"),
    ("evaluation_report", RUN_ROOT / "artifacts" / "evaluation_report.json"),
    ("compare_summary", RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json"),
    ("signoff_report", RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json"),
    ("workflow_log", RUN_ROOT / "logs" / "workflow_execution.log"),
    ("coverage_audit_log", RUN_ROOT / "logs" / "coverage_audit.log"),
]

STAGE_STATUS = {}
for name, path in check_items:
    exists = path.exists()
    mark = "✓" if exists else "✗"
    print(f"{mark} {name}: {path}")

STAGE_STATUS["embed"] = "ok" if (RUN_ROOT / "records" / "embed_record.json").exists() else "failed"
STAGE_STATUS["detect"] = "ok" if (RUN_ROOT / "records" / "detect_record.json").exists() else "failed"
STAGE_STATUS["calibrate"] = "ok" if (RUN_ROOT / "records" / "calibration_record.json").exists() else "failed"
STAGE_STATUS["evaluate"] = "ok" if (RUN_ROOT / "records" / "evaluate_record.json").exists() else "failed"
STAGE_STATUS["multi_protocol"] = "ok" if (RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json").exists() else "failed"
STAGE_STATUS["signoff"] = "ok" if (RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json").exists() else "failed"

coverage_log_path = RUN_ROOT / "logs" / "coverage_audit.log"
if coverage_log_path.exists():
    STAGE_STATUS["coverage_audit"] = "ok"
elif "COVERAGE_AUDIT_RESULT" in globals():
    STAGE_STATUS["coverage_audit"] = "ok"
else:
    STAGE_STATUS["coverage_audit"] = "not_run"

print("\n阶段状态汇总：")
print(STAGE_STATUS)

if "COVERAGE_AUDIT_RESULT" in globals():
    print("\ncoverage 审计摘要：")
    print({
        "result": COVERAGE_AUDIT_RESULT.get("result", "<absent>"),
        "missed_conditions": COVERAGE_AUDIT_RESULT.get("evidence", {}).get("missed_conditions", []),
        "extra_reported_conditions": COVERAGE_AUDIT_RESULT.get("evidence", {}).get("extra_reported_conditions", []),
    })
else:
    print("\ncoverage 审计摘要：not_run（可选步骤，建议执行第 5.5 步）")

## 第 9 步：验证工作流输出

检查工作流是否成功生成了所有必要的输出文件。

In [ ]:
import json

print("=" * 80)
print("验证工作流输出")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再验证输出")

print("\n[1/5] 检查记录文件...")
records_dir = RUN_ROOT / "records"
expected_records = [
    "embed_record.json",
    "detect_record.json",
    "calibration_record.json",
    "evaluate_record.json",
]

records_found = {}
for record_name in expected_records:
    record_path = records_dir / record_name
    exists = record_path.exists()
    records_found[record_name] = exists
    status = "✓" if exists else "✗"
    print(f"  {status} {record_name}")

print("\n[2/5] 检查产物文件...")
artifacts_dir = RUN_ROOT / "artifacts"
expected_artifacts = [
    "evaluation_report.json",
    "run_closure.json",
    "multi_protocol_evaluation/artifacts/protocol_compare/compare_summary.json",
    "signoff/signoff_report.json",
]

artifacts_found = {}
for artifact_name in expected_artifacts:
    artifact_path = artifacts_dir / artifact_name
    exists = artifact_path.exists()
    artifacts_found[artifact_name] = exists
    status = "✓" if exists else "✗"
    print(f"  {status} {artifact_name}")

print("\n[3/5] 校验 signoff 决策...")
signoff_decision = "<absent>"
signoff_report_path = artifacts_dir / "signoff" / "signoff_report.json"
if signoff_report_path.exists():
    with open(signoff_report_path, "r", encoding="utf-8") as f:
        signoff_report = json.load(f)
    signoff_decision = signoff_report.get("freeze_signoff_decision", "<absent>")
print(f"  freeze_signoff_decision: {signoff_decision}")

print("\n[4/5] 阶段状态汇总...")
if 'STAGE_STATUS' in globals():
    print(f"  阶段状态：{STAGE_STATUS}")
else:
    print("  未检测到 STAGE_STATUS（可能未执行第 5 步）")

print("\n[5/5] evaluation_report 语义锚点检查...")
evaluation_report_path = artifacts_dir / "evaluation_report.json"
semantic_ok = False
if evaluation_report_path.exists():
    with open(evaluation_report_path, "r", encoding="utf-8") as f:
        raw_eval_report = json.load(f)

    if isinstance(raw_eval_report, dict) and isinstance(raw_eval_report.get("evaluation_report"), dict):
        eval_report = raw_eval_report["evaluation_report"]
    else:
        eval_report = raw_eval_report

    required_anchor_fields = [
        "cfg_digest",
        "plan_digest",
        "thresholds_digest",
        "threshold_metadata_digest",
        "impl_digest",
        "fusion_rule_version",
        "attack_protocol_version",
        "attack_protocol_digest",
        "policy_path",
    ]

    def _read_anchor(report_obj, key):
        if isinstance(report_obj.get(key), str) and report_obj.get(key):
            return report_obj.get(key)
        anchors_obj = report_obj.get("anchors")
        if isinstance(anchors_obj, dict):
            value = anchors_obj.get(key)
            if isinstance(value, str) and value:
                return value
        return None

    missing = [
        field for field in required_anchor_fields
        if (_read_anchor(eval_report, field) is None or _read_anchor(eval_report, field) == "<absent>")
    ]
    report_type_ok = eval_report.get("report_type") == "evaluation_report"
    metrics_list_ok = isinstance(eval_report.get("metrics_by_attack_condition"), list)
    semantic_ok = report_type_ok and metrics_list_ok and (len(missing) == 0)

    print(f"  report_type_ok: {report_type_ok}")
    print(f"  metrics_by_attack_condition_is_list: {metrics_list_ok}")
    print(f"  missing_anchor_fields: {missing}")
else:
    print("  ✗ 缺失 evaluation_report.json，无法进行语义检查")

all_required_records = all(records_found.values())
all_required_artifacts = all(artifacts_found.values())
is_signoff_allow = (signoff_decision == "ALLOW_FREEZE")

print("\n验证结论：")
print(f"  records 完整：{all_required_records}")
print(f"  artifacts 完整：{all_required_artifacts}")
print(f"  signoff 通过：{is_signoff_allow}")
print(f"  evaluation_report 语义合格：{semantic_ok}")
print(f"  输出目录：{RUN_ROOT}")

## 第 10 步：结果打包和下载

将完整的 run_root 目录压缩并下载到本地计算机。

In [ ]:
import json
import shutil
from pathlib import Path

print("=" * 80)
print("打包和下载结果")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步")

required_files = list(globals().get("REQUIRED_PACKAGE_FILES", []))
if not required_files:
    required_files = [
        RUN_ROOT / "records" / "embed_record.json",
        RUN_ROOT / "records" / "detect_record.json",
        RUN_ROOT / "records" / "calibration_record.json",
        RUN_ROOT / "records" / "evaluate_record.json",
        RUN_ROOT / "artifacts" / "evaluation_report.json",
        RUN_ROOT / "artifacts" / "run_closure.json",
        RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json",
        RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json",
    ]

missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    print("\n✗ 检测到缺失必需产物，禁止打包：")
    for item in missing_files:
        print(f"  - {item}")
    raise RuntimeError("必需产物不完整，请先修复 workflow 后再打包")

signoff_report_path = RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json"
signoff_decision = "<absent>"
if signoff_report_path.exists():
    with open(signoff_report_path, "r", encoding="utf-8") as f:
        signoff_obj = json.load(f)
    signoff_decision = signoff_obj.get("freeze_signoff_decision", "<absent>")

package_manifest = {
    "run_root": str(RUN_ROOT),
    "profile": globals().get("ACTIVE_WORKFLOW_PROFILE", "paper_full_cuda"),
    "signoff_profile": globals().get("ACTIVE_SIGNOFF_PROFILE", "paper"),
    "signoff_decision": signoff_decision,
    "required_files": [str(p.relative_to(RUN_ROOT)) for p in required_files],
}
manifest_path = RUN_ROOT / "artifacts" / "package_manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(package_manifest, f, indent=2, ensure_ascii=False)
print(f"✓ 已写入打包清单：{manifest_path}")

ARCHIVE_NAME = f"ceg_wm_run_root_{CONFIG_CHOICE}_paper_full_cuda"
ARCHIVE_PATH = WORK_DIR / f"{ARCHIVE_NAME}.zip"

print(f"\n压缩目录结构...")
print(f"  源目录：{RUN_ROOT}")
print(f"  目标文件：{ARCHIVE_PATH.name}")

try:
    archive_without_ext = str(WORK_DIR / ARCHIVE_NAME)
    shutil.make_archive(
        archive_without_ext,
        "zip",
        RUN_ROOT.parent,
        RUN_ROOT.name
    )

    if ARCHIVE_PATH.exists():
        size_mb = ARCHIVE_PATH.stat().st_size / (1024 * 1024)
        print("✓ 压缩成功")
        print(f"  文件大小：{size_mb:.2f} MB")
        print(f"  完整路径：{ARCHIVE_PATH}")
    else:
        print("✗ 压缩失败（文件不存在）")
        ARCHIVE_PATH = None

except Exception as e:
    print(f"✗ 压缩过程出错：{e}")
    ARCHIVE_PATH = None

if ARCHIVE_PATH and ARCHIVE_PATH.exists():
    print(f"\n下载压缩包...")

    if IN_COLAB:
        from google.colab import files
        try:
            files.download(str(ARCHIVE_PATH))
            print(f"✓ 文件已下载：{ARCHIVE_PATH.name}")
            print("  查看浏览器的下载文件夹")
        except Exception as e:
            print(f"✗ 下载失败：{e}")
    else:
        print("本地环境提示：")
        print(f"  压缩包位置：{ARCHIVE_PATH}")
        print("  可以直接从文件系统访问或下载")
else:
    print("✗ 无法下载（压缩包创建失败）")

print("\n✓ 打包和下载步骤完成")

## 最后：执行总结报告

生成详细的执行总结。

In [ ]:
import json
from datetime import datetime

print("\n" + "=" * 80)
print("执行总结")
print("=" * 80)

if 'records_found' not in globals():
    records_found = {}
if 'artifacts_found' not in globals():
    artifacts_found = {}

runtime_config_used = str(ACTIVE_CONFIG_FILE) if 'ACTIVE_CONFIG_FILE' in globals() else str(CONFIG_FILE)
workflow_profile = globals().get("ACTIVE_WORKFLOW_PROFILE", "paper_full_cuda")
signoff_profile = globals().get("ACTIVE_SIGNOFF_PROFILE", "paper")

summary = {
    "timestamp": datetime.now().isoformat(),
    "environment": "Google Colab" if IN_COLAB else "Local Jupyter",
    "system_info": {
        "total_memory_gb": round(psutil.virtual_memory().total / (1024**3), 1),
        "available_memory_gb": round(psutil.virtual_memory().available / (1024**3), 1),
        "cpu_count": psutil.cpu_count(),
    },
    "config_used": CONFIG_CHOICE,
    "runtime_config_used": runtime_config_used,
    "workflow_profile": workflow_profile,
    "signoff_profile": signoff_profile,
    "ceg_wm_root": str(CEG_WM_ROOT),
    "run_root": str(RUN_ROOT),
    "records": {
        "embed": records_found.get("embed_record.json", False),
        "detect": records_found.get("detect_record.json", False),
        "calibrate": records_found.get("calibration_record.json", False),
        "evaluate": records_found.get("evaluate_record.json", False),
    },
    "artifacts": {
        "evaluation_report": artifacts_found.get("evaluation_report.json", False),
        "run_closure": artifacts_found.get("run_closure.json", False),
        "compare_summary": artifacts_found.get("multi_protocol_evaluation/artifacts/protocol_compare/compare_summary.json", False),
        "signoff_report": artifacts_found.get("signoff/signoff_report.json", False),
    },
    "signoff_decision": globals().get("SIGNOFF_DECISION", "<absent>"),
    "archive": {
        "created": ARCHIVE_PATH is not None and ARCHIVE_PATH.exists(),
        "path": str(ARCHIVE_PATH) if ARCHIVE_PATH else None,
        "size_mb": round(ARCHIVE_PATH.stat().st_size / (1024 * 1024), 2) if (ARCHIVE_PATH and ARCHIVE_PATH.exists()) else None,
    },
}

print("\n元数据：")
print(f"  执行时间：{summary['timestamp']}")
print(f"  运行环境：{summary['environment']}")
print(f"  配置：{summary['config_used']}")
print(f"  运行配置：{summary['runtime_config_used']}")
print(f"  workflow profile：{summary['workflow_profile']}")
print(f"  signoff profile：{summary['signoff_profile']}")
print(f"  signoff decision：{summary['signoff_decision']}")

print("\n系统资源：")
for key, value in summary['system_info'].items():
    print(f"  {key}: {value}")

print("\n生成的记录文件：")
for stage, exists in summary['records'].items():
    status = "✓" if exists else "✗"
    print(f"  {status} {stage}_record.json")

print("\n生成的工件文件：")
for artifact, exists in summary['artifacts'].items():
    status = "✓" if exists else "✗"
    print(f"  {status} {artifact}")

print("\n压缩包信息：")
if summary['archive']['created']:
    print(f"  ✓ 已创建：{summary['archive']['path']}")
    print(f"  大小：{summary['archive']['size_mb']} MB")
else:
    print("  ✗ 压缩包创建失败")

summary_file = RUN_ROOT / "execution_summary.json"
with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\n执行总结已保存到：{summary_file}")

print("\n" + "=" * 80)
print("✓ Colab 工作流执行完成！")
print("=" * 80)

## 配置选项参考

本 Notebook 第 5 步直接使用项目内的 `configs/paper_full_cuda.yaml`，并以 `paper_full_cuda` profile 执行。

| 配置项 | 值 | 说明 |
|---------|------|------|
| `workflow profile` | `paper_full_cuda` | 调用 `scripts/run_onefile_workflow.py` 的 profile |
| `cfg` | `configs/paper_full_cuda.yaml` | 仓库内正式 paper 配置（不再生成临时 colab yaml） |
| `model_id` | `stabilityai/stable-diffusion-3.5-medium` | 真实 SD3.5 模型 |
| `device` | `cuda` | 要求 Colab GPU Runtime |
| `signoff profile` | `paper` | 启用严格审计与签署路径 |

### 如需调整
- 如需修改模型或参数，请直接在项目配置文件中调整，或新建正式配置文件后在第 5 步改 `PROJECT_PAPER_CFG`。

## 输出结构参考

执行完成后，`run_root` 目录重点关注以下文件：

```
run_root/
├── records/
│   ├── embed_record.json
│   ├── detect_record.json
│   ├── calibration_record.json
│   └── evaluate_record.json
├── artifacts/
│   ├── evaluation_report.json
│   ├── run_closure.json
│   ├── package_manifest.json
│   ├── multi_protocol_evaluation/
│   │   └── artifacts/protocol_compare/compare_summary.json
│   └── signoff/
│       └── signoff_report.json
└── logs/
    ├── workflow_execution.log
    ├── audits_strict_rerun.log   # 第 6 步可选复跑生成
    └── signoff_rerun.log         # 第 7 步可选复跑生成
```

### 关键输出文件说明
- **evaluation_report.json**：评估阶段输出的核心报告。
- **run_closure.json**：流程闭包与状态信息。
- **multi_protocol.../compare_summary.json**：多协议对比摘要（paper 必需）。
- **signoff/signoff_report.json**：冻结签署决策报告。
- **package_manifest.json**：打包前写入的产物清单与签署决策快照。
- **workflow_execution.log**：第 5 步一键执行日志（排错优先查看）。